<a href="https://colab.research.google.com/github/giyuubin/group-project/blob/main/train_malware_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import joblib

# 1. A파트에서 만든 최적화된 데이터 불러오기
print("데이터를 불러옵니다...")
df = pd.read_csv('optimized_data.csv')

# 문제(X: 30개 핵심 API)와 정답(y: malware 여부) 분리
X = df.drop(columns=['malware'])
y = df['malware']

# 2. 훈련용(80%) / 테스트용(20%) 데이터 분할
# stratify=y 를 통해 정상/악성코드의 비율이 양쪽에 고르게 섞이도록 유도합니다.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"데이터 분할 완료! (훈련용: {len(X_train)}개 / 테스트용: {len(X_test)}개)\n")

# 3. 모델 2종 훈련 (랜덤 포레스트 vs 로지스틱 회귀)
print("인공지능 모델 2종을 학습시키는 중입니다. 잠시만 기다려주세요...")

# 3-1. 랜덤 포레스트 (배깅 앙상블 기법)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 3-2. 로지스틱 회귀 (선형 분류 기법)
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

# 4. 모델 성능 평가 함수 정의
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    print(f"========== [{name} 모델 평가 결과] ==========")
    print("1. 오차 행렬(Confusion Matrix):")
    print(confusion_matrix(y_test, y_pred))

    print("\n2. 주요 평가 지표:")
    print(f"- 정확도(Accuracy) : {accuracy_score(y_test, y_pred):.4f}")
    print(f"- 정밀도(Precision): {precision_score(y_test, y_pred):.4f}")
    print(f"- 재현율(Recall)   : {recall_score(y_test, y_pred):.4f}")
    print(f"- F1-Score         : {f1_score(y_test, y_pred):.4f}\n")

# 평가 실행
evaluate_model("랜덤 포레스트", rf_model, X_test, y_test)
evaluate_model("로지스틱 회귀", lr_model, X_test, y_test)

# 5. 완성된 최종 모델 저장
# 통상적으로 악성코드 탐지와 같은 복잡한 비선형 문제에서는 랜덤 포레스트의 성능이 우수합니다.
joblib.dump(rf_model, 'malware_detector.pkl')
print("✅ 실전에 투입할 '랜덤 포레스트' 모델이 'malware_detector.pkl' 파일로 안전하게 저장되었습니다!")